# LeetCode 650 — 2 Keys Keyboard bằng Python

Link đề: https://leetcode.com/problems/2-keys-keyboard/ · Độ khó: **Medium** · Chủ đề: Math, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def minSteps(self, n: int) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết lại theo hiểu biết về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (so với **BFS vét cạn trên mọi trạng thái** cho tất cả `n` từ `1` đến `1000`), nhưng nên đối chiếu lại đề gốc nếu muốn khớp từng chữ. Các ví dụ ngoài hai ví dụ chính của đề là ví dụ **tự dựng** (đối chiếu lại đề gốc nếu cần).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: màn hình ban đầu có đúng một chữ `A`; mỗi bước chỉ được **Copy All** (sao chép **toàn bộ** màn hình, không sao chép một phần) hoặc **Paste** (dán nội dung đã sao chép gần nhất). Tìm số bước **ít nhất** để màn hình có **đúng `n`** chữ `A`.
2. Nắm các công cụ Python: trạng thái là **cặp** `(độ dài màn hình, độ dài clipboard)`, `collections.deque` cho BFS, `dict`, `functools.cache`, phép chia lấy dư `%` / chia nguyên `//`, vòng lặp `while p * p <= n` để phân tích ra thừa số nguyên tố.
3. Đi từ **BFS trên mọi trạng thái (vét cạn)** → **quy hoạch động theo ước số** → **đệ quy có nhớ theo ước** → **tổng các thừa số nguyên tố** → **mô phỏng tham lam "copy khi chia hết"**.
4. Hiểu mấu chốt: một lần **Copy rồi dán `k` lần** nhân độ dài lên `k + 1` lần với chi phí `k + 1` bước. Vậy cần phân tích `n = f1·f2·...·fm` sao cho **tổng `f1 + f2 + ... + fm` nhỏ nhất**, và vì `a·b ≥ a + b` (với `a, b ≥ 2`) nên chia nhỏ tới **thừa số nguyên tố** luôn tốt nhất.
5. Tránh các lỗi kinh điển: Copy một lần rồi dán `n - 1` lần (đáp án `n`), quên thừa số nguyên tố còn dư lớn hơn `√n` khi phân tích, chỉ tách **một** lần thay vì đệ quy, và quên mất `clipboard` trong trạng thái BFS.

## 2. Đề bài & ví dụ

Trên màn hình một cuốn sổ chỉ có **một** ký tự `'A'`. Ở mỗi bước, bạn thực hiện **một** trong hai thao tác:

- **Copy All:** sao chép **tất cả** ký tự đang có trên màn hình (không được sao chép một phần).
- **Paste:** dán các ký tự đã sao chép **lần gần nhất**.

Cho số nguyên `n`, trả về **số thao tác ít nhất** để trên màn hình có **đúng `n`** ký tự `'A'`.

**Ví dụ 1**

- Input: `n = 3`
- Output: `3`
- Giải thích: ban đầu có một `'A'`. Bước 1: Copy All. Bước 2: Paste được `AA`. Bước 3: Paste được `AAA`.

**Ví dụ 2**

- Input: `n = 1`
- Output: `0`
- Giải thích: màn hình đã có đúng một `'A'`, không cần thao tác nào.

**Ràng buộc:**

- `1 <= n <= 1000`

Chú ý: **Paste khi clipboard còn trống** không có tác dụng (ban đầu clipboard trống nên bước đầu tiên có ích duy nhất là Copy). Sau Copy, nội dung clipboard luôn bằng **toàn bộ** màn hình lúc đó.

In [1]:
from typing import List, Optional
from collections import deque
from functools import cache
import heapq
import random

examples = [
    (3, 3),
    (1, 0),
]
extra_examples = [  # tự dựng, đã kiểm chứng bằng BFS vét cạn; nên đối chiếu lại đề gốc
    (2, 2),
    (4, 4),
    (6, 5),
    (9, 6),
    (12, 7),
    (1000, 21),
]
for n, expected in examples + extra_examples:
    print("n =", n, "-> kỳ vọng", expected)

n = 3 -> kỳ vọng 3
n = 1 -> kỳ vọng 0
n = 2 -> kỳ vọng 2
n = 4 -> kỳ vọng 4
n = 6 -> kỳ vọng 5
n = 9 -> kỳ vọng 6
n = 12 -> kỳ vọng 7
n = 1000 -> kỳ vọng 21


## 3. Kiến thức nền

### 3.1. Trạng thái là một **cặp** `(độ dài màn hình, độ dài clipboard)`

Để biết "Paste sẽ làm gì" ta cần biết **cả** màn hình có bao nhiêu `A` **và** clipboard đang giữ bao nhiêu `A`. Vì vậy trạng thái là cặp `(length, clip)`. Hai thao tác:

- **Copy:** `(length, clip) → (length, length)`.
- **Paste:** `(length, clip) → (length + clip, clip)`.

Ban đầu `(1, 0)`. Ta viết hàm `run_ops` chạy một chuỗi thao tác (`"C"` = Copy, `"P"` = Paste) để quan sát.

In [2]:
def run_ops(ops):
    """Chạy chuỗi thao tác, trả độ dài màn hình cuối cùng. 'C' = Copy All, 'P' = Paste."""
    length, clip = 1, 0
    for ch in ops:
        if ch == "C":
            clip = length
        else:
            length += clip
    return length


print(run_ops("CPP"))        # 3  (ví dụ 1)
print(run_ops("CPCP"))       # 4  : 1 -> C -> P (2) -> C (clip=2) -> P (4)
print(run_ops("CPCPP"))      # 6
print(run_ops("P"))          # 1  : Paste khi clipboard trống không làm gì

3
4
6
1


### 3.2. Ước số, phép chia lấy dư `%` và chia nguyên `//`

`a % b == 0` nghĩa là `b` **chia hết** `a` (`b` là **ước** của `a`). `a // b` là thương nguyên. Ví dụ ước của `12` là `1, 2, 3, 4, 6, 12`. Để tìm ước ta chỉ cần duyệt `i` tới `√n` (khi `i * i <= n`): mỗi ước `i` cho cả ước đối `n // i`.

In [3]:
n = 12
print(12 % 5, 12 // 5, 12 % 4, 12 // 4)           # 2 2 0 3
divs = []
i = 1
while i * i <= n:
    if n % i == 0:
        divs.append(i)
        if i != n // i:
            divs.append(n // i)
    i += 1
print(sorted(divs))                               # [1, 2, 3, 4, 6, 12]

2 2 0 3
[1, 2, 3, 4, 6, 12]


### 3.3. Phân tích ra thừa số nguyên tố

**Số nguyên tố** là số `≥ 2` chỉ chia hết cho `1` và chính nó. Mọi số `≥ 2` viết được **duy nhất** thành tích các số nguyên tố, ví dụ `12 = 2 · 2 · 3`. Thuật toán chia thử: với `p = 2, 3, 4, ...` **chia hết bao nhiêu lần có thể** (`while n % p == 0`), mỗi lần ghi nhận `p`; dừng khi `p * p > n`. Phần còn lại (nếu `> 1`) chính là **một thừa số nguyên tố lớn** còn sót.

In [4]:
def prime_factors(n):
    res = []
    p = 2
    while p * p <= n:
        while n % p == 0:
            res.append(p)
            n //= p
        p += 1
    if n > 1:
        res.append(n)          # thừa số nguyên tố còn dư (lớn hơn căn của phần còn lại)
    return res


for x in (12, 18, 97, 1000, 14):
    print(x, prime_factors(x))

12 [2, 2, 3]
18 [2, 3, 3]
97 [97]
1000 [2, 2, 2, 5, 5, 5]
14 [2, 7]


### 3.4. BFS (duyệt theo tầng) tìm số bước ít nhất

Khi mỗi bước có chi phí bằng nhau, **BFS** từ trạng thái đầu cho **số bước ít nhất** tới mọi trạng thái. Dùng `deque` làm hàng đợi (`append` thêm cuối, `popleft` lấy đầu) và `dict` `dist` ghi **khoảng cách** (số bước) tới mỗi trạng thái đã thăm.

In [5]:
# Ví dụ nhỏ: BFS trên đồ thị số, mỗi bước +1 hoặc *2, tìm số bước từ 1 tới 6
dist = {1: 0}
q = deque([1])
while q:
    u = q.popleft()
    for v in (u + 1, u * 2):
        if v <= 6 and v not in dist:
            dist[v] = dist[u] + 1
            q.append(v)
print(dist)                    # {1: 0, 2: 1, 3: 2, 4: 2, 6: 3, 5: 3}

{1: 0, 2: 1, 3: 2, 4: 2, 6: 3, 5: 3}


### 3.5. Ghi nhớ kết quả: `functools.cache`

`@cache` đặt trên một hàm đệ quy sẽ **nhớ** kết quả theo tham số; lần gọi lại cùng tham số trả ngay. Rất hợp để cài đặt quy hoạch động "từ trên xuống" (top-down).

In [6]:
calls = 0


@cache
def fib(k):
    global calls
    calls += 1
    return k if k < 2 else fib(k - 1) + fib(k - 2)


print(fib(30), "số lần thực sự tính:", calls)       # 832040, 31

832040 số lần thực sự tính: 31


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Thử `n = 6`. Cách ngây thơ: Copy một lần rồi dán thật nhiều: `C P P P P P` (1 Copy + 5 Paste) = **6** bước. Có cách nào ngắn hơn?

- `C P` (màn hình `2`), rồi `C P P` (clipboard `2`, dán hai lần: `2 → 4 → 6`): tổng `2 + 3 = ` **5** bước.
- `C P P` (màn hình `3`), rồi `C P` (clipboard `3`, dán một lần: `3 → 6`): tổng `3 + 2 = ` **5** bước.

Quan sát **then chốt**: một lần **Copy rồi dán `k` lần** biến độ dài `L` thành `L · (k + 1)` và tốn `1 + k = (k + 1)` bước. Nghĩa là **nhân độ dài lên `f` lần tốn đúng `f` bước**.

| Cách tách `6` thành tích | Tổng chi phí |
|---|---|
| `6` | `6` |
| `2 · 3` | `2 + 3 = 5` |
| `3 · 2` | `3 + 2 = 5` |

Với `n = 12`: `12` → 12; `2 · 6` → 8; `3 · 4` → 7; `2 · 2 · 3` → **7**. Càng tách nhỏ thì tổng càng giảm (đến một mức): bài toán trở thành **"tách `n` thành tích các số `≥ 2` sao cho tổng nhỏ nhất"**.

In [7]:
print(run_ops("C" + "P" * 5))            # 6 với 6 bước
print(run_ops("CP" + "CPP"))             # 6 với 5 bước
print(run_ops("CPP" + "CP"))             # 6 với 5 bước


def factorizations(n, start=2):
    """Mọi cách viết n = f1 * f2 * ... * fk với 2 <= f1 <= f2 <= ... (n >= 2)."""
    res = [[n]]
    f = start
    while f * f <= n:
        if n % f == 0:
            for rest in factorizations(n // f, f):
                res.append([f] + rest)
        f += 1
    return res


for x in (6, 12):
    for fac in factorizations(x):
        print(x, "=", " * ".join(map(str, fac)), "-> chi phí", sum(fac))

6
6
6
6 = 6 -> chi phí 6
6 = 2 * 3 -> chi phí 5
12 = 12 -> chi phí 12
12 = 2 * 6 -> chi phí 8
12 = 2 * 2 * 3 -> chi phí 7
12 = 3 * 4 -> chi phí 7


## 5. Cách 1 — Brute Force / baseline: BFS trên mọi trạng thái `(length, clip)`

**Ý tưởng.** Không dùng bất kỳ nhận xét toán học nào. Coi mỗi trạng thái `(length, clip)` là một đỉnh; từ mỗi đỉnh có tối đa hai cạnh (Copy, Paste). BFS từ `(1, 0)` cho số bước ít nhất tới mọi trạng thái. Ta dừng ngay khi **sinh ra** một trạng thái có `length == n` (BFS đảm bảo đó là số bước nhỏ nhất). Vì độ dài chỉ tăng nên ta **bỏ qua** các trạng thái có `length > n`.

In [8]:
def min_steps_bfs_demo(n):
    if n == 1:
        return 0
    dist = {(1, 0): 0}
    q = deque([(1, 0)])
    while q:
        length, clip = q.popleft()
        d = dist[(length, clip)]
        for nxt, name in (((length, length), "Copy"), ((length + clip, clip), "Paste")):
            if nxt[0] > n or nxt in dist:
                continue
            dist[nxt] = d + 1
            print(f"  {(length, clip)} --{name}--> {nxt}   (bước {d + 1})")
            if nxt[0] == n:
                return d + 1
            q.append(nxt)


print("n = 6 ->", min_steps_bfs_demo(6))

  (1, 0) --Copy--> (1, 1)   (bước 1)
  (1, 1) --Paste--> (2, 1)   (bước 2)
  (2, 1) --Copy--> (2, 2)   (bước 3)
  (2, 1) --Paste--> (3, 1)   (bước 3)
  (2, 2) --Paste--> (4, 2)   (bước 4)
  (3, 1) --Copy--> (3, 3)   (bước 4)
  (3, 1) --Paste--> (4, 1)   (bước 4)
  (4, 2) --Copy--> (4, 4)   (bước 5)
  (4, 2) --Paste--> (6, 2)   (bước 5)
n = 6 -> 5


In [9]:
def min_steps_bfs(n):
    """Cách 1: BFS vét cạn trên trạng thái (độ dài, clipboard)."""
    if n == 1:
        return 0
    dist = {(1, 0): 0}
    q = deque([(1, 0)])
    while q:
        length, clip = q.popleft()
        d = dist[(length, clip)]
        for nxt in ((length, length), (length + clip, clip)):
            if nxt[0] > n or nxt in dist:
                continue
            if nxt[0] == n:
                return d + 1
            dist[nxt] = d + 1
            q.append(nxt)


for n, expected in examples + extra_examples:
    got = min_steps_bfs(n)
    print(n, got, got == expected)

3 3 True
1 0 True
2 2 True
4 4 True
6 5 True
9 6 True
12 7 True
1000 21 True


**Độ phức tạp.** Có tối đa `O(n²)` trạng thái (`length ≤ n`, `clip ≤ length`), mỗi trạng thái tốn `O(1)`. Time `O(n²)`, Space `O(n²)`. Với `n = 1000` là cỡ `5·10^5` trạng thái: vẫn chạy được, nhưng **lãng phí**.

**Nút thắt.** Ta không dùng cấu trúc đặc biệt của bài: chỉ có những trạng thái "ngay sau một lần Copy" mới quan trọng, và độ dài sau mỗi khối `Copy + Paste...` là **bội** của độ dài trước đó. Nhờ đó ta chỉ cần quan tâm tới **các ước của `n`**.

## 6. Ý tưởng thuật toán / bước đột phá

**Phân tách thành khối.** Mọi chuỗi thao tác có ích đều có dạng: `C P^k1 · C P^k2 · ... · C P^km` (mỗi khối là một Copy rồi dán `k_i ≥ 1` lần):

- Bước đầu **phải** là Copy (clipboard đang trống, Paste vô ích).
- Hai Copy **liên tiếp** vô ích (Copy sau ghi đè Copy trước, màn hình không đổi) nên bỏ bớt được.
- Một Copy **không có Paste** theo sau (ở cuối) cũng vô ích.

Khối `C P^k` đưa độ dài từ `L` lên `L · (k + 1)` và tốn `k + 1` bước. Đặt `f = k + 1 ≥ 2`: khối **nhân độ dài lên `f` lần với chi phí `f`**.

Vậy: `n = f1 · f2 · ... · fm` với mọi `f_i ≥ 2`, chi phí `f1 + f2 + ... + fm`; cần **cực tiểu hóa tổng**. Có hai hệ quả:

1. **Quy hoạch động theo ước:** `dp[n] = min` trên mọi ước `d < n` của `n` của `dp[d] + n / d` (đã có độ dài `d`, nhân lên `n / d` lần với chi phí `n / d`). `dp[1] = 0`.
2. **Chỉ cần thừa số nguyên tố:** nếu một thừa số `f = a·b` với `a, b ≥ 2` thì thay `f` bằng hai thừa số `a, b` làm tổng không tăng, vì `a·b − a − b = (a − 1)(b − 1) − 1 ≥ 0`. Lặp lại tới khi chỉ còn số nguyên tố. Do phân tích thành số nguyên tố là **duy nhất**, đáp án là **tổng các thừa số nguyên tố** (tính cả bội) của `n`, và bằng `0` khi `n = 1`.

## 7. Các cách tiếp theo

### Cách 2 — Quy hoạch động bottom-up theo ước số

**Giải thích từng đoạn.**

- `dp[i]` là số bước ít nhất để có đúng `i` ký tự. Khởi tạo `dp[i] = i` (Copy một lần rồi dán `i - 1` lần, đi từ độ dài `1`), `dp[1] = 0`.
- Với mỗi `j` từ `2` tới `n` (khi tới `j`, `dp[j]` **đã tối ưu** vì mọi ước nhỏ hơn của `j` đã được xử lý trước): với mỗi **bội** `k = 2j, 3j, ...`, cập nhật `dp[k] = min(dp[k], dp[j] + k // j)`.
- Đáp án: `dp[n]`.

Đây là cách duyệt kiểu "sàng": thay vì tìm ước của từng `k`, ta từ mỗi `j` đẩy tới các bội của nó.

In [10]:
def min_steps_dp_demo(n):
    dp = list(range(n + 1))          # dp[i] = i (Copy 1 lần + dán i-1 lần); dp[0] không dùng
    dp[1] = 0
    for j in range(2, n + 1):
        for k in range(2 * j, n + 1, j):
            if dp[j] + k // j < dp[k]:
                print(f"  dp[{k}] cập nhật {dp[k]} -> {dp[j] + k // j}  (từ dp[{j}] = {dp[j]} và nhân {k // j} lần)")
                dp[k] = dp[j] + k // j
    return dp


print(min_steps_dp_demo(12))

  dp[6] cập nhật 6 -> 5  (từ dp[2] = 2 và nhân 3 lần)
  dp[8] cập nhật 8 -> 6  (từ dp[2] = 2 và nhân 4 lần)
  dp[10] cập nhật 10 -> 7  (từ dp[2] = 2 và nhân 5 lần)
  dp[12] cập nhật 12 -> 8  (từ dp[2] = 2 và nhân 6 lần)
  dp[9] cập nhật 9 -> 6  (từ dp[3] = 3 và nhân 3 lần)
  dp[12] cập nhật 8 -> 7  (từ dp[3] = 3 và nhân 4 lần)
[0, 0, 2, 3, 4, 5, 5, 7, 6, 6, 7, 11, 7]


In [11]:
class Solution2:
    def minSteps(self, n: int) -> int:
        dp = list(range(n + 1))
        dp[1] = 0
        for j in range(2, n + 1):
            for k in range(2 * j, n + 1, j):
                dp[k] = min(dp[k], dp[j] + k // j)
        return dp[n]


for n, expected in examples + extra_examples:
    assert Solution2().minSteps(n) == expected
print("Cách 2 đúng trên các ví dụ")

Cách 2 đúng trên các ví dụ


**Độ phức tạp Cách 2:** hai vòng lặp kiểu sàng: tổng số cập nhật `n/2 + n/3 + ... ≈ n ln n` ⇒ Time `O(n log n)`, Space `O(n)`.

---

### Cách 3 — Đệ quy có nhớ theo ước (top-down)

**Giải thích từng đoạn.**

- `dfs(m)` = số bước ít nhất để có `m` ký tự. `dfs(1) = 0`.
- Với `m > 1`: mặc định `ans = m` (một khối duy nhất: Copy + dán `m - 1` lần). Rồi thử mọi cặp ước `(i, m // i)` với `2 ≤ i ≤ √m`: hoặc khối **cuối** nhân lên `i` lần từ độ dài `m // i` (chi phí `dfs(m // i) + i`), hoặc khối cuối nhân lên `m // i` lần từ độ dài `i` (chi phí `dfs(i) + m // i`). Nhờ xét cả hai chiều, mọi cách tách đều được bao trùm.
- `@cache` để không tính lại.

In [12]:
def min_steps_memo_demo(n):
    memo_calls = []

    @cache
    def dfs(m):
        memo_calls.append(m)
        if m == 1:
            return 0
        ans = m
        i = 2
        while i * i <= m:
            if m % i == 0:
                ans = min(ans, dfs(m // i) + i, dfs(i) + m // i)
            i += 1
        return ans

    res = dfs(n)
    print("các giá trị m thực sự được tính:", sorted(set(memo_calls)))
    return res


print(min_steps_memo_demo(12))

các giá trị m thực sự được tính: [2, 3, 4, 6, 12]
7


In [13]:
class Solution3:
    def minSteps(self, n: int) -> int:
        @cache
        def dfs(m: int) -> int:
            if m == 1:
                return 0
            ans = m
            i = 2
            while i * i <= m:
                if m % i == 0:
                    ans = min(ans, dfs(m // i) + i, dfs(i) + m // i)
                i += 1
            return ans

        return dfs(n)


for n, expected in examples + extra_examples:
    assert Solution3().minSteps(n) == expected
print("Cách 3 đúng trên các ví dụ")

Cách 3 đúng trên các ví dụ


**Độ phức tạp Cách 3:** các giá trị `m` thực sự được tính chỉ là **ước của `n`** (và vài ước của ước); mỗi `m` tốn `O(√m)` để duyệt ước ⇒ Time khoảng `O(d(n) · √n)` với `d(n)` là số ước của `n` (rất nhỏ: tối đa `32` khi `n ≤ 1000`), Space `O(d(n))` cho bảng nhớ cộng ngăn xếp đệ quy `O(log n)`.

---

### Cách 4 — Tổng các thừa số nguyên tố (cách chuẩn, ngắn nhất)

**Ý tưởng.** Theo mục 6, đáp án là **tổng các thừa số nguyên tố** (tính cả bội) của `n`.

**Giải thích từng đoạn.**

- `ans = 0`, `p = 2`.
- `while p * p <= n:` thử `p` làm thừa số: `while n % p == 0:` cộng `p` vào `ans` rồi chia `n` cho `p`.
- Tăng `p`. Khi thoát vòng lặp, nếu `n > 1` thì `n` là **một số nguyên tố** còn dư ⇒ `ans += n`.
- Với `n = 1`: không vào vòng nào, `n > 1` sai ⇒ `ans = 0`. Đúng đề.

In [14]:
def min_steps_factor_demo(n):
    ans = 0
    p = 2
    m = n
    while p * p <= m:
        while m % p == 0:
            ans += p
            m //= p
            print(f"  chia hết cho {p}: ans = {ans}, còn lại {m}")
        p += 1
    if m > 1:
        ans += m
        print(f"  phần dư nguyên tố {m}: ans = {ans}")
    return ans


print(min_steps_factor_demo(12))
print(min_steps_factor_demo(14))

  chia hết cho 2: ans = 2, còn lại 6
  chia hết cho 2: ans = 4, còn lại 3
  phần dư nguyên tố 3: ans = 7
7
  chia hết cho 2: ans = 2, còn lại 7
  phần dư nguyên tố 7: ans = 9
9


In [15]:
class Solution4:
    def minSteps(self, n: int) -> int:
        ans = 0
        p = 2
        while p * p <= n:
            while n % p == 0:
                ans += p
                n //= p
            p += 1
        if n > 1:
            ans += n
        return ans


for n, expected in examples + extra_examples:
    assert Solution4().minSteps(n) == expected
print("Cách 4 đúng trên các ví dụ")

Cách 4 đúng trên các ví dụ


**Độ phức tạp Cách 4:** vòng `while p * p <= n` chạy tối đa `√n` lần ⇒ Time `O(√n)`, Space `O(1)`.

---

### Cách 5 — Mô phỏng tham lam "Copy ngay khi độ dài hiện tại chia hết `n`"

**Ý tưởng.** Chạy thẳng quá trình thao tác. Biến `cur` (độ dài), `clip` (clipboard), `ops` (số bước). Lặp khi `cur < n`:

- Nếu `n % cur == 0` (độ dài hiện tại là **ước** của `n`) thì **Copy** (`clip = cur`, `ops += 1`).
- Luôn **Paste** (`cur += clip`, `ops += 1`).

**Vì sao đúng?** Sau khi Copy ở độ dài `c`, ta dán lặp lại: độ dài đi qua `c·2, c·3, ...`. Lần Copy **kế tiếp** xảy ra ở `c·k` đầu tiên (`k ≥ 2`) mà `c·k` chia hết `n`, tức `k` là **ước nhỏ nhất `≥ 2` của `n / c`**, chính là **thừa số nguyên tố nhỏ nhất** còn lại. Vậy mô phỏng này **tách `n` thành các thừa số nguyên tố nhỏ nhất trước**, đúng như Cách 4.

In [16]:
def min_steps_greedy_demo(n):
    cur, clip, ops = 1, 0, 0
    print(f"khởi tạo: cur={cur} clip={clip} ops={ops}")
    while cur < n:
        if n % cur == 0:
            clip = cur
            ops += 1
            print(f"  {n} % {cur} == 0 -> Copy: clip={clip} ops={ops}")
        cur += clip
        ops += 1
        print(f"  Paste: cur={cur} ops={ops}")
    return ops


print(min_steps_greedy_demo(12))

khởi tạo: cur=1 clip=0 ops=0
  12 % 1 == 0 -> Copy: clip=1 ops=1
  Paste: cur=2 ops=2
  12 % 2 == 0 -> Copy: clip=2 ops=3
  Paste: cur=4 ops=4
  12 % 4 == 0 -> Copy: clip=4 ops=5
  Paste: cur=8 ops=6
  Paste: cur=12 ops=7
7


In [17]:
class Solution5:
    def minSteps(self, n: int) -> int:
        cur, clip, ops = 1, 0, 0
        while cur < n:
            if n % cur == 0:
                clip = cur
                ops += 1
            cur += clip
            ops += 1
        return ops


for n, expected in examples + extra_examples:
    assert Solution5().minSteps(n) == expected
print("Cách 5 đúng trên các ví dụ")

Cách 5 đúng trên các ví dụ


**Độ phức tạp Cách 5:** mỗi lần lặp `cur` tăng ít nhất `1` nên tối đa `n` lần lặp (tệ nhất khi `n` là số nguyên tố: dán từng một) ⇒ Time `O(n)`, Space `O(1)`.

## 8. TRACE từng bước

**Bảng quy hoạch động `dp` cho `n = 12` (Cách 2):**

| `i` | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| `dp[i]` | 0 | 2 | 3 | 4 | 5 | 5 | 7 | 6 | 6 | 7 | 11 | 7 |
| Phân tích | | `2` | `3` | `2·2` | `5` | `2·3` | `7` | `2·2·2` | `3·3` | `2·5` | `11` | `2·2·3` |

(Ví dụ `dp[12] = dp[6] + 2 = 7` hoặc `dp[4] + 3 = 7`.)

**Trace Cách 4 với `n = 12`:** `p = 2`: `12 % 2 = 0` ⇒ `ans = 2`, `n = 6`; `6 % 2 = 0` ⇒ `ans = 4`, `n = 3`; `3 % 2 ≠ 0`. `p = 3`: `3 * 3 = 9 > 3` thoát vòng. `n = 3 > 1` ⇒ `ans = 7`. Dãy thao tác thực sự (mỗi thừa số `f` là `C` rồi `P` lặp `f - 1` lần): `CP · CP · CPP` = 7 bước, độ dài `1 → 2 → 4 → 12`.

Hàm `trace_factor` dưới đây in từng bước và **dựng luôn dãy thao tác**, rồi dùng `run_ops` kiểm lại độ dài.

In [18]:
def build_ops_from_factors(n):
    """Dựng dãy thao tác ngắn nhất: mỗi thừa số nguyên tố f cho 'C' + 'P' * (f - 1)."""
    ops = ""
    for f in prime_factors(n):
        ops += "C" + "P" * (f - 1)
    return ops


def trace_factor(n):
    print(f"n = {n}")
    ans = 0
    length = 1
    ops = ""
    for f in prime_factors(n):
        block = "C" + "P" * (f - 1)
        ans += f
        length *= f
        ops += block
        print(f"  thừa số {f}: khối {block!r} (chi phí {f}) -> độ dài {length}, tổng bước {ans}")
    print(f"  dãy thao tác: {ops!r}, run_ops = {run_ops(ops)}")
    assert run_ops(ops) == n and len(ops) == ans
    return ans


assert trace_factor(12) == 7
assert trace_factor(18) == 8
assert trace_factor(1) == 0

n = 12
  thừa số 2: khối 'CP' (chi phí 2) -> độ dài 2, tổng bước 2
  thừa số 2: khối 'CP' (chi phí 2) -> độ dài 4, tổng bước 4
  thừa số 3: khối 'CPP' (chi phí 3) -> độ dài 12, tổng bước 7
  dãy thao tác: 'CPCPCPP', run_ops = 12
n = 18
  thừa số 2: khối 'CP' (chi phí 2) -> độ dài 2, tổng bước 2
  thừa số 3: khối 'CPP' (chi phí 3) -> độ dài 6, tổng bước 5
  thừa số 3: khối 'CPP' (chi phí 3) -> độ dài 18, tổng bước 8
  dãy thao tác: 'CPCPPCPP', run_ops = 18
n = 1
  dãy thao tác: '', run_ops = 1


**Trace BFS cho `n = 6` (Cách 1)** theo tầng (khoảng cách `d`): `d=0`: `(1,0)`. `d=1`: Copy `(1,1)`. `d=2`: Paste `(2,1)`. `d=3`: từ `(2,1)`: Copy `(2,2)`, Paste `(3,1)`. `d=4`: từ `(2,2)`: Paste `(4,2)`; từ `(3,1)`: Copy `(3,3)`, Paste `(4,1)`. `d=5`: từ `(4,2)`: Paste `(6,2)` đạt `n = 6`. Vậy `5` bước, đúng bảng ở mục 4.

In [19]:
assert min_steps_bfs(6) == 5
print(min_steps_bfs_demo(6))

  (1, 0) --Copy--> (1, 1)   (bước 1)
  (1, 1) --Paste--> (2, 1)   (bước 2)
  (2, 1) --Copy--> (2, 2)   (bước 3)
  (2, 1) --Paste--> (3, 1)   (bước 3)
  (2, 2) --Paste--> (4, 2)   (bước 4)
  (3, 1) --Copy--> (3, 3)   (bước 4)
  (3, 1) --Paste--> (4, 1)   (bước 4)
  (4, 2) --Copy--> (4, 4)   (bước 5)
  (4, 2) --Paste--> (6, 2)   (bước 5)
5


## 9. Code nộp LeetCode

Ta nộp **Cách 4** (tổng thừa số nguyên tố): `O(√n)`, `O(1)` bộ nhớ, ngắn nhất. Không cần `import` gì.

In [20]:
class Solution:
    def minSteps(self, n: int) -> int:
        ans = 0
        p = 2
        while p * p <= n:
            while n % p == 0:         # mỗi thừa số nguyên tố p tốn p bước (Copy + dán p-1 lần)
                ans += p
                n //= p
            p += 1
        if n > 1:                     # còn dư một thừa số nguyên tố lớn
            ans += n
        return ans


for n, expected in examples + extra_examples:
    assert Solution().minSteps(n) == expected
print("Code nộp đúng trên 8 ví dụ")

Code nộp đúng trên 8 ví dụ


## 10. Vì sao thuật toán đúng

**(a) Mọi chuỗi tối ưu có dạng khối.** Một chuỗi thao tác chỉ có hai loại; hãy loại bỏ các thao tác vô ích mà không làm tăng số bước: (i) Paste khi clipboard trống (không đổi gì); (ii) Copy ngay sau một Copy (hai Copy liên tiếp cho cùng clipboard, bỏ cái trước); (iii) Copy cuối cùng không có Paste sau nó. Sau khi bỏ, chuỗi là `C P^{k1} C P^{k2} ... C P^{km}` với mọi `k_i ≥ 1`. Khối `C P^{k}` đưa độ dài `L` thành `L + k·L = L (k + 1)`.

**(b) Quy về bài toán tích-tổng.** Đặt `f_i = k_i + 1 ≥ 2`. Độ dài cuối là `f1 · f2 · ... · fm = n` và tổng số bước `Σ (1 + k_i) = Σ f_i`. Ngược lại, mọi cách viết `n = f1 · ... · fm` (`f_i ≥ 2`) cho một chuỗi thao tác. Vậy đáp án là **min của tổng `Σ f_i` trên mọi cách phân tích `n` thành tích các số `≥ 2`** (và `0` khi `n = 1`, không có khối nào).

**(c) Tách tới nguyên tố là tốt nhất.** Nếu một thừa số `f = a·b` với `a, b ≥ 2`, thay `f` bằng hai thừa số `a, b`: tích không đổi, tổng đổi `a + b − ab = −(a − 1)(b − 1) + 1 ≤ 0` (vì `(a−1)(b−1) ≥ 1`) tức không tăng. Lặp lại cho tới khi mọi thừa số nguyên tố. Phân tích thành số nguyên tố là **duy nhất** (định lý cơ bản của số học) nên tổng đó là một giá trị xác định `sopfr(n)` (sum of prime factors with repetition) và là **tối ưu**: mọi cách phân tích khác đều có thể tách thành đúng bộ thừa số này mà tổng không tăng.

**(d) Các cách còn lại.** Cách 2 và Cách 3 tính trực tiếp "min của tổng" bằng quy hoạch động theo ước, nên đúng theo (b). Trong Cách 3 ta duyệt `i ≤ √m` và thử cả `dfs(m // i) + i` lẫn `dfs(i) + m // i`, nên mọi cặp ước `(i, m // i)` đều được xét. Cách 5 mô phỏng việc "tách thừa số nguyên tố nhỏ nhất trước" (giải thích ở mục 7), cho cùng `sopfr(n)`. Cách 1 (BFS) không dựa vào các nhận xét trên và là **oracle** để kiểm chứng: ở mục 12 ta so cả bốn cách với BFS cho **mọi `n` từ `1` đến `1000`**.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. BFS trên `(length, clip)` | `O(n²)` | `O(n²)` | vét cạn, dùng làm oracle |
| 2. DP bottom-up theo bội | `O(n log n)` | `O(n)` | sàng kiểu "đẩy tới bội" |
| 3. Đệ quy có nhớ theo ước | `O(d(n) · √n)` | `O(d(n))` | `d(n)` = số ước, rất nhỏ |
| 4. Tổng thừa số nguyên tố (nộp) | `O(√n)` | `O(1)` | ngắn nhất, nhanh nhất |
| 5. Mô phỏng tham lam | `O(n)` | `O(1)` | tệ nhất khi `n` nguyên tố |

Với `n ≤ 1000`, mọi cách trừ BFS đều gần như tức thì; chọn Cách 4 vì ngắn và có thể mở rộng cho `n` rất lớn (`10^12`: `√n = 10^6` phép chia).

## 12. Bộ test

### 12.1. Ví dụ đề, biên và các giá trị đặc biệt

Gồm: `n = 1` (đáp án `0`), số nguyên tố (đáp án `= n`), lũy thừa của `2` (đáp án `2·k` với `n = 2^k`), bình phương nguyên tố (`p²` cho `2p`), `n = 1000`.

In [21]:
solvers = {
    "Cách 1 BFS": min_steps_bfs,
    "Cách 2 DP": Solution2().minSteps,
    "Cách 3 đệ quy nhớ": Solution3().minSteps,
    "Cách 4 thừa số nguyên tố": Solution4().minSteps,
    "Cách 5 mô phỏng": Solution5().minSteps,
    "Code nộp": Solution().minSteps,
}

special = [
    (1, 0), (2, 2), (3, 3), (5, 5), (7, 7), (11, 11), (997, 997),      # số nguyên tố -> n
    (4, 4), (8, 6), (16, 8), (64, 12), (512, 18),                      # 2^k -> 2k
    (9, 6), (25, 10), (49, 14), (961, 62),                             # p^2 -> 2p
    (6, 5), (12, 7), (30, 10), (210, 17), (1000, 21),                  # hợp số
]
for n, expected in examples + extra_examples + special:
    for name, f in solvers.items():
        if name == "Cách 1 BFS" and n > 300:
            continue          # BFS chậm với n lớn, kiểm riêng ở 12.2
        assert f(n) == expected, (name, n, f(n), expected)
print("Ví dụ, biên và giá trị đặc biệt đều đúng")

Ví dụ, biên và giá trị đặc biệt đều đúng


### 12.2. Oracle BFS cho **mọi** `n` từ `1` đến `1000`

Một lần BFS từ `(1, 0)` trên mọi trạng thái có `length ≤ 1000` cho ngay `best[n]` = số bước ít nhất để có đúng `n` (lần đầu BFS gặp độ dài `n`). Bảng này **độc lập hoàn toàn** với các nhận xét toán học và được dùng làm đáp án chuẩn.

In [22]:
def min_steps_oracle_table(N):
    """BFS một lần từ (1, 0), trả best[n] cho mọi n <= N."""
    dist = {(1, 0): 0}
    q = deque([(1, 0)])
    best = [None] * (N + 1)
    best[1] = 0
    while q:
        length, clip = q.popleft()
        d = dist[(length, clip)]
        for nxt in ((length, length), (length + clip, clip)):
            if nxt[0] > N or nxt in dist:
                continue
            dist[nxt] = d + 1
            if best[nxt[0]] is None:
                best[nxt[0]] = d + 1
            q.append(nxt)
    return best


best = min_steps_oracle_table(1000)
assert all(best[n] is not None for n in range(1, 1001))
print("best[1..12] =", best[1:13])           # [0, 2, 3, 4, 5, 5, 7, 6, 6, 7, 11, 7]
for n in range(1, 1001):
    for name, f in solvers.items():
        if name == "Cách 1 BFS" and n > 150:
            continue
        assert f(n) == best[n], (name, n, f(n), best[n])
print("Cả 6 cách khớp oracle BFS với mọi n = 1..1000 (Cách 1 chỉ kiểm tới 150)")

best[1..12] = [0, 2, 3, 4, 5, 5, 7, 6, 6, 7, 11, 7]


Cả 6 cách khớp oracle BFS với mọi n = 1..1000 (Cách 1 chỉ kiểm tới 150)


### 12.3. So BFS riêng lẻ với oracle bảng, và tính chất

Cách 1 chạy từng `n` (có dừng sớm) phải trùng với bảng một-lần. Ta cũng kiểm tra vài tính chất toán học của hàm `f(n)` = đáp án: `f(1) = 0`; `f(p) = p` với `p` nguyên tố; `f(a·b) ≤ f(a) + f(b)` (ghép hai chuỗi thao tác); `f(a·b) = f(a) + f(b)` (đúng vì tính cộng của `sopfr`).

In [23]:
rng = random.Random(650)
for _ in range(60):
    n = rng.randint(1, 120)
    assert min_steps_bfs(n) == best[n]

def is_prime(m):
    return m >= 2 and all(m % d for d in range(2, int(m ** 0.5) + 1))

f = Solution().minSteps
assert f(1) == 0
for n in range(2, 1001):
    if is_prime(n):
        assert f(n) == n
    assert f(n) <= n
for _ in range(500):
    a = rng.randint(1, 31)
    b = rng.randint(1, 31)
    assert f(a * b) == f(a) + f(b)       # a*b <= 961 <= 1000
print("Các tính chất: f(1)=0, f(nguyên tố)=nguyên tố, f(a*b)=f(a)+f(b), f(n) <= n đều đúng")

Các tính chất: f(1)=0, f(nguyên tố)=nguyên tố, f(a*b)=f(a)+f(b), f(n) <= n đều đúng


### 12.4. Ngẫu nhiên có seed và độ lớn của kết quả

Chọn `n` ngẫu nhiên trong `[1, 1000]`, kiểm tra các cách với oracle; và kiểm tra thêm cho `n` **lớn hơn ràng buộc** (cho Cách 4 và Cách 2/3, so sánh nhau): Cách 4 với `n` cỡ `10^9` (cho thấy tại sao `O(√n)` đáng giá).

In [24]:
rng = random.Random(6500)
for _ in range(300):
    n = rng.randint(1, 1000)
    exp = best[n]
    for name, g in solvers.items():
        if name == "Cách 1 BFS":
            continue
        assert g(n) == exp, (name, n)

for n in (10 ** 9 + 7, 2 ** 29, 3 ** 18, 720720, 999999937):
    # Cách 4 và Cách 3 so với nhau (Cách 2 sẽ tốn bộ nhớ O(n), bỏ qua)
    a = Solution().minSteps(n)
    if n <= 10 ** 6:
        assert a == Solution3().minSteps(n)
    print(n, "->", a)
assert Solution().minSteps(2 ** 29) == 58
assert Solution().minSteps(3 ** 18) == 54

1000000007 -> 1000000007
536870912 -> 58
387420489 -> 54
720720 -> 50
999999937 -> 999999937


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Copy một lần rồi dán `n - 1` lần (đáp án luôn `n`)

Nghĩ "Copy rồi dán thật nhiều" là đủ. Nhưng với số **hợp số**, ta có thể **Copy lại** giữa chừng để nhân độ dài nhanh hơn.

In [25]:
def sai_copy_mot_lan(n):
    return 0 if n == 1 else n          # 1 Copy + (n - 1) Paste


for n in (6, 12, 1000):
    print(n, "sai:", sai_copy_mot_lan(n), "| đúng:", Solution().minSteps(n))
assert sai_copy_mot_lan(6) != Solution().minSteps(6)

6 sai: 6 | đúng: 5
12 sai: 12 | đúng: 7
1000 sai: 1000 | đúng: 21


**Vì sao sai:** `6` có thể làm bằng `CP CPP` (5 bước) thay vì `CPPPPP` (6 bước). **Sửa:** phân tích `n` thành các thừa số và cộng chi phí từng khối.

### Lỗi 2: Quên thừa số nguyên tố lớn còn dư sau vòng `while p * p <= n`

Vòng lặp dừng khi `p * p > n`. Lúc đó `n` còn lại có thể là một **số nguyên tố lớn hơn `p`** (ví dụ `14 = 2 · 7`: sau khi chia `2`, còn `7`, và `3 * 3 > 7` nên dừng). Nếu quên cộng nó thì thiếu.

In [26]:
def sai_quen_du(n):
    ans = 0
    p = 2
    while p * p <= n:
        while n % p == 0:
            ans += p
            n //= p
        p += 1
    return ans                          # SAI: thiếu `if n > 1: ans += n`


print("n = 7  sai:", sai_quen_du(7), "| đúng:", Solution().minSteps(7))       # 0 vs 7
print("n = 14 sai:", sai_quen_du(14), "| đúng:", Solution().minSteps(14))     # 2 vs 9
assert sai_quen_du(7) != 7 and sai_quen_du(14) != 9

n = 7  sai: 0 | đúng: 7
n = 14 sai: 2 | đúng: 9


**Vì sao sai:** thừa số nguyên tố lớn nhất của `n` có thể lớn hơn `√n`; chỉ có **tối đa một** thừa số như vậy và nó còn lại trong `n`. **Sửa:** sau vòng lặp, `if n > 1: ans += n`.

### Lỗi 3: Chỉ tách **một lần** thay vì tách đệ quy

Nghĩ rằng đáp án là `min` của `i + n // i` với mọi ước `i` (chỉ chia thành **hai** khối). Nhưng có thể chia thành **nhiều** khối còn tốt hơn.

In [27]:
def sai_tach_mot_lan(n):
    if n == 1:
        return 0
    return min([i + n // i for i in range(2, int(n ** 0.5) + 1) if n % i == 0] + [n])


bad = [n for n in range(1, 101) if sai_tach_mot_lan(n) != best[n]]
print("các n đầu tiên bị sai:", bad[:8])             # bắt đầu từ 18
n = bad[0]
print(f"n = {n}: sai = {sai_tach_mot_lan(n)}, đúng = {Solution().minSteps(n)} (phân tích {prime_factors(n)})")
assert bad and bad[0] == 18

các n đầu tiên bị sai: [18, 24, 27, 30, 32, 36, 40, 42]
n = 18: sai = 9, đúng = 8 (phân tích [2, 3, 3])


**Vì sao sai:** `18 = 2 · 3 · 3` cho `2 + 3 + 3 = 8`, nhưng tách thành hai khối chỉ được `min(2 + 9, 3 + 6) = 9`. **Sửa:** đệ quy (`dfs(n // i) + i`) hoặc dùng thẳng thừa số nguyên tố.

### Lỗi 4: BFS đánh dấu đã thăm chỉ theo **độ dài**, quên clipboard

Hai trạng thái cùng độ dài nhưng clipboard khác nhau là **khác nhau** (Paste cho kết quả khác). Nếu gom theo độ dài thì Copy (không đổi độ dài) luôn bị coi là "đã thăm" và bị bỏ.

In [28]:
def sai_bfs_chi_theo_do_dai(n):
    if n == 1:
        return 0
    seen = {1}
    q = deque([(1, 0, 0)])
    while q:
        length, clip, d = q.popleft()
        for nl, nc in ((length, length), (length + clip, clip)):
            if nl > n or nl in seen:           # SAI: phải kiểm tra (nl, nc)
                continue
            if nl == n:
                return d + 1
            seen.add(nl)
            q.append((nl, nc, d + 1))
    return None


print("n = 2 sai:", sai_bfs_chi_theo_do_dai(2), "| đúng:", Solution().minSteps(2))      # None vs 2
print("n = 6 sai:", sai_bfs_chi_theo_do_dai(6), "| đúng:", Solution().minSteps(6))
assert sai_bfs_chi_theo_do_dai(2) is None

n = 2 sai: None | đúng: 2
n = 6 sai: None | đúng: 5


**Vì sao sai:** trạng thái đầu `(1, 0)` có độ dài `1`; Copy sinh `(1, 1)` cùng độ dài `1` nên bị coi là đã thăm và không bao giờ được xét; BFS không thể tiến. **Sửa:** dùng **cặp** `(length, clip)` làm khóa của `dist`/`seen`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "thao tác nhân đôi/nhân lên theo bội", "chi phí tỉ lệ với hệ số nhân" ⇒ nghĩ ngay tới **ước số, phân tích thừa số**.
- **Gom thao tác thành khối** (Copy + các Paste) để thấy cấu trúc **nhân**; từ đó dùng hàm "tích → tổng".
- **`a·b ≥ a + b` khi `a, b ≥ 2`:** nên khi cần **cực tiểu hóa tổng với tích cố định**, hãy tách thành **số nguyên tố** (cực đại hóa số thừa số, mỗi thừa số nhỏ nhất).
- **Quy hoạch động theo ước:** `dp[n] = min dp[d] + chi_phí(n / d)`; duyệt ước tới `√n` hoặc "đẩy tới bội" kiểu sàng.
- **BFS trên trạng thái nhỏ** là cách kiểm chứng đáng tin cậy cho mọi giả thuyết toán học; hãy dựng bảng oracle một lần.
- **Số nguyên tố:** đáp án `= n` (không có cách nào ngắn hơn).
- **Thử `n = 12, 18, 24`:** đủ khác nhau để lộ lỗi "chỉ tách một lần" hay "quên phần dư".
- Bài liên quan: **651. 4 Keys Keyboard** (Ctrl-A/C/V, tối đa hóa số `A`), **Perfect Squares** (DP kiểu min), **Integer Break** (tích → tối đa).

## 15. Biến thể

1. **Trả về dãy thao tác tối ưu** (không chỉ số bước): dựng từ thừa số nguyên tố (`build_ops_from_factors`).
2. **Chi phí khác nhau:** Copy tốn `c_copy`, Paste tốn `c_paste`. Khi đó tách nguyên tố **chưa chắc tối ưu** (ví dụ Copy rất đắt), nên dùng DP theo ước: `dp[k] = min dp[j] + c_copy + (k / j − 1) · c_paste`.
3. **Chỉ được Copy một phần / có Cut:** mất cấu trúc "nhân"; cần BFS hoặc DP phức tạp hơn.
4. **651. 4 Keys Keyboard:** tối đa hóa số `A` với `n` lần bấm (Ctrl-A, Ctrl-C, Ctrl-V).
5. **`n` rất lớn (`10^12`):** Cách 4 vẫn nhanh (`√n = 10^6`).

In [29]:
def min_cost_dp(n, c_copy, c_paste):
    """Biến thể 2: chi phí Copy = c_copy, Paste = c_paste. dp theo ước (dạng đẩy tới bội)."""
    INF = float("inf")
    dp = [INF] * (n + 1)
    dp[1] = 0
    for j in range(1, n + 1):
        if dp[j] == INF:
            continue
        for k in range(2 * j, n + 1, j):
            dp[k] = min(dp[k], dp[j] + c_copy + (k // j - 1) * c_paste)
    return dp[n]


def min_cost_dijkstra(n, c_copy, c_paste):
    """Oracle cho biến thể 2: Dijkstra trên trạng thái (length, clip)."""
    best = {(1, 0): 0}
    heap = [(0, 1, 0)]
    while heap:
        d, length, clip = heapq.heappop(heap)
        if best.get((length, clip), float("inf")) < d:
            continue
        if length == n:
            return d
        for nxt, w in (((length, length), c_copy), ((length + clip, clip), c_paste)):
            if nxt[0] > n:
                continue
            nd = d + w
            if nd < best.get(nxt, float("inf")):
                best[nxt] = nd
                heapq.heappush(heap, (nd, nxt[0], nxt[1]))
    return None


assert min_cost_dp(12, 1, 1) == 7
print([min_cost_dp(12, c, 1) for c in (1, 5, 20)])     # Copy càng đắt thì càng ít Copy hơn
rng_v = random.Random(6501)
for _ in range(120):
    n = rng_v.randint(1, 60)
    cc, cp = rng_v.randint(1, 9), rng_v.randint(1, 9)
    assert min_cost_dp(n, cc, cp) == min_cost_dijkstra(n, cc, cp), (n, cc, cp)
print("Biến thể chi phí tùy ý: DP theo ước khớp Dijkstra")

[7, 15, 31]
Biến thể chi phí tùy ý: DP theo ước khớp Dijkstra


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS `(length, clip)` | Duyệt mọi trạng thái, số bước ít nhất theo tầng | `O(n²)` | `O(n²)` | Làm oracle, khi chưa thấy cấu trúc |
| 2. DP bottom-up theo bội | `dp[k] = min dp[j] + k/j` từ mỗi `j` tới các bội | `O(n log n)` | `O(n)` | Cần đáp án cho **nhiều** `n`, hoặc chi phí tùy ý |
| 3. Đệ quy có nhớ theo ước | `dfs(m) = min dfs(m/i) + i` | `O(d(n)·√n)` | `O(d(n))` | Dễ viết từ công thức, ít trạng thái |
| 4. Tổng thừa số nguyên tố | Tách tới nguyên tố, cộng các thừa số | `O(√n)` | `O(1)` | **Nộp bài**, `n` lớn |
| 5. Mô phỏng tham lam | Copy khi độ dài chia hết `n`, luôn Paste | `O(n)` | `O(1)` | Muốn chạy "thật" theo đề, dễ kiểm bằng mắt |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Solution().minSteps(1)`, `Solution().minSteps(7)`, `Solution().minSteps(16)`, `Solution().minSteps(12)`, và `run_ops("CPCPP")`.

**Bài 2 (tính tay).** Với `n = 18`: phân tích thành thừa số nguyên tố, tính số bước ít nhất, **viết ra dãy thao tác** (dạng chuỗi `C`/`P`) và kiểm tra bằng `run_ops`. Có cách nào cũng 8 bước nhưng khác dãy không?

**Bài 3 (điền TODO).** Hoàn thành `min_steps_dp_todo(n)` theo Cách 2.

In [30]:
def min_steps_dp_todo(n):
    # TODO 1: dp = list(range(n + 1)); dp[1] = 0   (dp[i] = i: Copy 1 lần + dán i-1 lần)
    # TODO 2: for j in range(2, n + 1): for k in range(2 * j, n + 1, j): dp[k] = min(dp[k], dp[j] + k // j)
    # TODO 3: return dp[n]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên chưa đúng).

**Bài 4 (tự viết lại).** Không nhìn code mẫu, viết `min_steps_sopfr(n)` bằng phân tích thừa số nguyên tố (Cách 4); nhớ xử lý `n = 1` và thừa số nguyên tố dư.

**Bài 5 (biến thể / mini challenge).** (a) Viết `build_ops(n)` trả về một **dãy thao tác ngắn nhất** (chuỗi `"C"`/`"P"`) để có đúng `n` ký tự; kiểm tra `run_ops(build_ops(n)) == n` và độ dài bằng đáp án. (b) Viết `min_cost(n, c_copy, c_paste)` cho chi phí Copy và Paste khác nhau (DP theo ước); kiểm tra với `min_cost_dijkstra` ở mục 15.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `0`; `7` (số nguyên tố); `8` (`2·2·2·2`); `7` (`2·2·3`); `6` (`CP CPP`: `1 → 2 → 6`). Bài 3: bám sát mục 7 (Cách 2).
- **Mức 2 (vừa):** Bài 2: `18 = 2 · 3 · 3` ⇒ `2 + 3 + 3 = 8`. Dãy: `"CP" + "CPP" + "CPP"` ⇒ độ dài `1 → 2 → 6 → 18`. Thứ tự các thừa số có thể hoán vị (`3, 2, 3` hoặc `3, 3, 2`) vẫn cho 8 bước. Bài 4: nhớ `if n > 1: ans += n` sau vòng `while p * p <= n`.
- **Mức 3 (sâu):** Bài 5(a): với mỗi thừa số nguyên tố `f` của `n`, nối `"C" + "P" * (f - 1)`. Bài 5(b): `dp[1] = 0`, từ mỗi `j` có `dp[j]` hữu hạn, với mỗi bội `k = m·j` (`m ≥ 2`) cập nhật `dp[k] = min(dp[k], dp[j] + c_copy + (m - 1) * c_paste)`; duyệt `j` tăng dần để `dp[j]` đã tối ưu khi dùng.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert Solution().minSteps(1) == 0
assert Solution().minSteps(7) == 7
assert Solution().minSteps(16) == 8
assert Solution().minSteps(12) == 7
assert run_ops("CPCPP") == 6

# Bài 2
assert prime_factors(18) == [2, 3, 3]
assert Solution().minSteps(18) == 8
ops18 = "CP" + "CPP" + "CPP"
assert run_ops(ops18) == 18 and len(ops18) == 8
assert run_ops("CPP" + "CP" + "CPP") == 18 and len("CPP" + "CP" + "CPP") == 8     # thứ tự khác, cũng 8 bước


# Bài 3
def min_steps_dp_ans(n):
    dp = list(range(n + 1))
    dp[1] = 0
    for j in range(2, n + 1):
        for k in range(2 * j, n + 1, j):
            dp[k] = min(dp[k], dp[j] + k // j)
    return dp[n]


# Bài 4
def min_steps_sopfr(n):
    ans = 0
    p = 2
    while p * p <= n:
        while n % p == 0:
            ans += p
            n //= p
        p += 1
    if n > 1:
        ans += n
    return ans


# Bài 5
def build_ops(n):
    ops = ""
    for f in prime_factors(n):
        ops += "C" + "P" * (f - 1)
    return ops


def min_cost(n, c_copy, c_paste):
    INF = float("inf")
    dp = [INF] * (n + 1)
    dp[1] = 0
    for j in range(1, n + 1):
        if dp[j] == INF:
            continue
        for k in range(2 * j, n + 1, j):
            dp[k] = min(dp[k], dp[j] + c_copy + (k // j - 1) * c_paste)
    return dp[n]


for n in range(1, 1001):
    assert min_steps_dp_ans(n) == min_steps_sopfr(n) == best[n]
    ops = build_ops(n)
    assert run_ops(ops) == n and len(ops) == best[n]
rng5 = random.Random(6502)
for _ in range(100):
    n = rng5.randint(1, 60)
    cc, cp = rng5.randint(1, 9), rng5.randint(1, 9)
    assert min_cost(n, cc, cp) == min_cost_dijkstra(n, cc, cp)
assert min_cost(12, 1, 1) == 7
print("Tất cả đáp án bài tập đều đúng.")

Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tổng các thừa số nguyên tố)

```python
class Solution:
    def minSteps(self, n: int) -> int:
        ans = 0
        p = 2
        while p * p <= n:
            while n % p == 0:      # mỗi thừa số nguyên tố p: Copy + (p - 1) Paste = p bước
                ans += p
                n //= p
            p += 1
        if n > 1:                  # thừa số nguyên tố dư (> căn của phần còn lại)
            ans += n
        return ans
```

**Checklist**

- [ ] Nhớ cấu trúc khối: **Copy rồi dán `k` lần = nhân `k + 1` lần với chi phí `k + 1`**?
- [ ] `n = 1` trả `0`?
- [ ] Sau vòng `while p * p <= n` có `if n > 1: ans += n` (thừa số nguyên tố dư)?
- [ ] Với số nguyên tố `p`, đáp án là `p` (không có cách ngắn hơn)?
- [ ] Không dừng ở "Copy một lần rồi dán `n - 1` lần" (sai với hợp số)?
- [ ] Nếu dùng DP/đệ quy: duyệt đủ ước (`dp[j] + k // j`) và `dp[1] = 0`?
- [ ] Đã thử `n = 1, 2, 3, 6, 12, 18, 1000` (đáp án `0, 2, 3, 5, 7, 8, 21`)?
- [ ] Skeleton đúng `class Solution` / `def minSteps(self, n: int) -> int`?